# Reachy inverse kinematics test
IK only: give a hand pose (position + orientation), get joint angles back.
Frame: origin in the torso, **x forward, y to Reachy's left, z up**, units = metres. Joint angles are degrees.

Run top to bottom. Sections 1-4 never move the arm. Section 5 only moves if `DRY_RUN = False`.
Moves go through `joint_limits.safe_goto` when `joint_limits.json` exists (record it with `record_joint_limits.ipynb`).

In [ ]:
from reachy_sdk import ReachySDK
from reachy_sdk.trajectory import goto
from reachy_sdk.trajectory.interpolation import InterpolationMode
from findIP import get_reachy_ip
import joint_limits as jl
import numpy as np, time

np.set_printoptions(precision=3, suppress=True)
reachy = ReachySDK(host=get_reachy_ip())

ARM_NAME = 'l_arm'
ARM = reachy.l_arm
CHAIN = list(ARM.kinematics_chain.values())   # the 7 arm joints (gripper excluded)
print([j.name for j in CHAIN])

try:
    LIMITS = jl.load_limits()
    print('joint limits loaded for:', sorted(LIMITS))
except FileNotFoundError as e:
    LIMITS = None
    print('WARNING:', e, '\n-> plans are only checked for big joint jumps')

In [ ]:
def current_q():
    return [j.present_position for j in CHAIN]

def rot_x(deg):
    c, s = np.cos(np.radians(deg)), np.sin(np.radians(deg))
    return np.array([[1, 0, 0], [0, c, -s], [0, s, c]])

def rot_y(deg):
    c, s = np.cos(np.radians(deg)), np.sin(np.radians(deg))
    return np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]])

def rot_z(deg):
    c, s = np.cos(np.radians(deg)), np.sin(np.radians(deg))
    return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])

def make_pose(xyz, R=np.eye(3)):
    """4x4 target pose from a position (m) and a 3x3 rotation."""
    T = np.eye(4)
    T[:3, :3] = R
    T[:3, 3] = xyz
    return T

def describe(T, label=''):
    x, y, z = T[:3, 3]
    print(f'{label} position (m): x={x:.3f}  y={y:.3f}  z={z:.3f}')
    print('rotation:\n', T[:3, :3])

def pose_error(A, B):
    """Position error (mm) and rotation error (deg) between two 4x4 poses."""
    pos_mm = np.linalg.norm(A[:3, 3] - B[:3, 3]) * 1000
    R = A[:3, :3].T @ B[:3, :3]
    ang = np.degrees(np.arccos(np.clip((np.trace(R) - 1) / 2, -1, 1)))
    return pos_mm, ang

def solve(T, q0=None, verbose=True):
    """IK + FK check. Returns (q, pos_mm, rot_deg) or (None, None, None) if IK fails."""
    q0 = current_q() if q0 is None else q0
    try:
        q = ARM.inverse_kinematics(T, q0=q0)
    except ValueError as e:
        if verbose:
            print('IK failed:', e)
        return None, None, None
    pos_mm, rot_deg = pose_error(T, ARM.forward_kinematics(joints_position=q))
    if verbose:
        print('IK q:', np.round(q, 1))
        print(f'FK of IK answer is off by {pos_mm:.1f} mm, {rot_deg:.2f} deg')
    return q, pos_mm, rot_deg

def limit_problems(q, q_now=None):
    """Joint-limit problems for a plan (empty list = OK, or None if no limits file)."""
    if LIMITS is None:
        return None
    q_now = current_q() if q_now is None else q_now
    q_res = [jl.resolve(j.name, a, c, LIMITS) for j, a, c in zip(CHAIN, q, q_now)]
    return jl.violations(CHAIN, q_res, q_now, LIMITS)

## 1. IK on the pose the hand is in right now (no motion)
The arm goes limp. Put it somewhere comfortable in front of Reachy, elbow bent.
IK should land on (almost) the same pose. The angles may differ: 7 joints means more than one answer for a pose.

In [ ]:
reachy.turn_off(ARM_NAME)

In [ ]:
q_now = current_q()
T_now = ARM.forward_kinematics()
describe(T_now, 'hand')
print('present q:', np.round(q_now, 1))
q_ik, _, _ = solve(T_now, q0=q_now)
print('angle diff vs present (deg):', np.round(np.array(q_ik) - np.array(q_now), 1))

## 2. Does the starting guess `q0` matter? (no motion)
Same target, different seeds. Different seeds can give different elbow postures for the same hand pose.
The one closest to the present posture is the one you want to execute, which is why the other notebooks pass `q0=current_q()`.

In [ ]:
T_target = T_now.copy()
seeds = {
    'present': q_now,
    'zeros':   [0.0] * len(CHAIN),
    'random1': list(np.random.default_rng(1).uniform(-60, 60, len(CHAIN))),
    'random2': list(np.random.default_rng(2).uniform(-60, 60, len(CHAIN))),
}
for name, q0 in seeds.items():
    q, pos_mm, rot_deg = solve(T_target, q0=q0, verbose=False)
    if q is None:
        print(f'{name:8s} IK failed')
        continue
    dist = np.abs(np.array(q) - np.array(q_now)).max()
    print(f'{name:8s} err {pos_mm:5.1f} mm {rot_deg:5.2f} deg | max joint diff from present {dist:6.1f} deg | q={np.round(q, 0)}')

## 3. Reachability map (no motion)
Grid of positions in front of the left shoulder, all with the current hand orientation.
`#` = reachable and accurate, `~` = IK answered but is off by more than `TOL_MM`, `.` = failed, `!` = reachable but breaks joint limits.
Use this to pick table targets that are comfortably inside the reachable area.

In [ ]:
XS = np.arange(0.15, 0.51, 0.05)     # forward
YS = np.arange(-0.10, 0.41, 0.05)    # left
Z = -0.25                            # height of the slice (m); change to look at other heights
TOL_MM = 10
R_keep = T_now[:3, :3]

q_seed = q_now
print(f'z = {Z:.2f} m, rows = x (forward), cols = y (left)')
print('      ' + ' '.join(f'{y:5.2f}' for y in YS))
for x in XS[::-1]:
    row = []
    for y in YS:
        q, pos_mm, _ = solve(make_pose([x, y, Z], R_keep), q0=q_seed, verbose=False)
        if q is None:
            c = '.'
        elif pos_mm > TOL_MM:
            c = '~'
        else:
            probs = limit_problems(q, q_now)
            c = '!' if probs else '#'
        row.append(f'{c:>5s}')
    print(f'{x:5.2f} ' + ' '.join(row))

## 4. Plan a target with a chosen orientation (no motion)
Here you set the orientation yourself instead of borrowing the current one.
`R_GRIP` below is the Pollen docs' "gripper pointing forward" orientation for Reachy (gripper z axis along the torso x axis).
Tweak it with `rot_x/rot_y/rot_z` (degrees) to tilt or roll the hand, e.g. `R_GRIP @ rot_z(90)`.
Read real `TARGET_XYZ` values from section 1 / the map in section 3.

In [ ]:
R_GRIP = np.array([[0, 0, -1],
                   [0, 1,  0],
                   [1, 0,  0]])
TARGET_XYZ = np.array([0.30, 0.20, -0.25])   # example only: replace with your own measured point
MAX_JOINT_JUMP = 60                           # refuse plans that swing any joint more than this (deg)

T_goal = make_pose(TARGET_XYZ, R_GRIP)
describe(T_goal, 'goal')

q_start = current_q()
q_goal, pos_mm, rot_deg = solve(T_goal, q0=q_start)
plan_ok = q_goal is not None
if plan_ok:
    jump = np.abs(np.array(q_goal) - np.array(q_start))
    for j, a, b, d in zip(CHAIN, q_start, q_goal, jump):
        print(f'{j.name:18s} {a:7.1f} -> {b:7.1f}   ({d:5.1f})')
    probs = limit_problems(q_goal, q_start)
    if pos_mm > 10 or rot_deg > 5:
        print('WARNING: IK did not really reach this pose (probably at the edge of the workspace)'); plan_ok = False
    if jump.max() > MAX_JOINT_JUMP:
        print('WARNING: big joint swing; move the arm closer to the goal posture by hand and re-run'); plan_ok = False
    if probs:
        print('WARNING: joint limits:\n  ' + '\n  '.join(probs)); plan_ok = False
print('plan OK' if plan_ok else 'plan REJECTED')

## 5. Execute the plan and measure accuracy
`turn_on` makes the arm stiff where it is, then it moves to the goal and back to where it started.

In [ ]:
DRY_RUN = True   # set to False when the plan above looks sane

def move_to(q_target, duration=2.0):
    if LIMITS is not None:
        jl.safe_goto(CHAIN, q_target, duration=duration, limits=LIMITS)   # raises before moving if unsafe
    else:
        goto({j: a for j, a in zip(CHAIN, q_target)}, duration=duration,
             interpolation_mode=InterpolationMode.MINIMUM_JERK)

if DRY_RUN:
    print('dry run: nothing moved')
elif not plan_ok:
    print('plan was rejected in section 4: not moving')
else:
    reachy.turn_on(ARM_NAME)
    move_to(q_goal, 3.0)
    time.sleep(0.5)
    T_reached = ARM.forward_kinematics()
    pos_mm, rot_deg = pose_error(T_goal, T_reached)
    print(f'reached goal within {pos_mm:.1f} mm, {rot_deg:.2f} deg')
    describe(T_reached, 'reached')
    move_to(q_start, 3.0)

In [ ]:
# Done: let the arm down gently (same arm we used)
reachy.turn_off_smoothly(ARM_NAME)